In [9]:
import pandas as pd

In [10]:
columns = ['ts', 'uid', 'id.orig_h', 'id.orig_p', 'id.resp_h', 'id.resp_p',
           'proto', 'service', 'duration', 'orig_bytes', 'resp_bytes',
           'conn_state', 'local_orig', 'local_resp', 'missed_bytes', 'history',
           'orig_pkts', 'orig_ip_bytes', 'resp_pkts', 'resp_ip_bytes',
           'tunnel_parents', 'ip_proto']

In [11]:
df = pd.read_csv("../data/raw/zeek-captures/beacon_conn.log",sep='\t',comment='#', names = columns)

In [16]:
df['conn_state']

0     OTH
1     OTH
2      S0
3      S0
4     SHR
     ... 
80    OTH
81    SHR
82    SHR
83    SHR
84    OTH
Name: conn_state, Length: 85, dtype: object

In [12]:
print(df.shape)
df.head()

(85, 22)


,ts,uid,id.orig_h,id.orig_p,id.resp_h,id.resp_p,proto,service,duration,orig_bytes,...,local_orig,local_resp,missed_bytes,history,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,tunnel_parents,ip_proto
0,1.790988e+09,CxdbMj4U97n1WAxnm7,fe80::20c:29ff:fe6c:6d6c,5353,ff02::fb,5353,udp,-,-,-,...,T,F,0,C,0,0,0,0,-,17
1,1.790988e+09,CppJk51WOITOEcSk94,192.168.150.129,5353,224.0.0.251,5353,udp,-,-,-,...,T,T,0,C,0,0,0,0,-,17
2,1.790988e+09,CMQdmOQ6dna2h8E97,192.168.150.1,50692,224.0.0.251,5353,udp,dns,-,-,...,T,T,0,D,1,73,0,0,-,17
3,1.790988e+09,C0tQa02W7XqnbcOjQd,192.168.150.1,53735,239.255.255.250,1900,udp,-,3.023189,680,...,T,T,0,D,4,792,0,0,-,17
4,1.790988e+09,CRuFYw3jKyOaRtErt7,192.168.150.129,50487,8.8.8.8,53,udp,dns,0.037343,0,...,T,F,0,Cd,0,0,1,422,-,17


In [35]:
target_ips = ['104.20.23.154', '172.66.147.243']
working_df = df[(df['conn_state'] == 'SHR')].sort_values(by='ts')


In [36]:
working_df['time_delta'] = filtered_df.groupby(["id.orig_h", "id.resp_h"])['ts'].diff()

In [37]:
stats = working_df.groupby(["id.orig_h", "id.resp_h"])['time_delta'].agg(['std','count'])
print(stats)

                                       std  count
id.orig_h       id.resp_h                        
192.168.150.129 104.20.23.154    40.366375      5
                172.66.147.243   53.842090      5
                192.168.150.254        NaN      0
                8.8.8.8                NaN      0
                91.189.91.64           NaN      0
                91.189.91.98           NaN      0


In [29]:
filtered_df.head()

,ts,uid,id.orig_h,id.orig_p,id.resp_h,id.resp_p,proto,service,duration,orig_bytes,...,local_resp,missed_bytes,history,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,tunnel_parents,ip_proto,time_delta
28,1.790988e+09,C2SZIW2NWFrYzrklW3,192.168.150.129,59736,104.20.23.154,80,tcp,-,0.090855,0,...,F,0,^hCadfA,1,40,5,1140,-,6,NaN
32,1.790988e+09,CXJwisIsShBrTC2p8,192.168.150.129,37826,104.20.23.154,80,tcp,-,0.080168,0,...,F,0,^hCadfA,1,40,5,1140,-,6,30.102212
35,1.790988e+09,CGpn5H3trwL1AMtJcc,192.168.150.129,52546,172.66.147.243,80,tcp,-,0.078765,0,...,F,0,^hCadfA,1,40,5,1140,-,6,NaN
39,1.790988e+09,CCMDRyiAuARMP1Ac2,192.168.150.129,50262,172.66.147.243,80,tcp,-,0.079912,0,...,F,0,^hCadfA,1,40,5,1140,-,6,30.086668
42,1.790988e+09,C6gzNF30mR5OhnibD8,192.168.150.129,34438,172.66.147.243,80,tcp,-,0.077226,0,...,F,0,^hCadfA,1,40,5,1140,-,6,30.093216


In [30]:
print("mean gap:", filtered_df['time_delta'].mean())
print("std gap:", filtered_df['time_delta'].std())

mean gap: 51.17068190574646
std gap: 44.97465764647095


"Baseline with IP-grouping: beacon fragmented across two CDN IPs (std 40 / 53), indistinguishable from noise. Motivates domain-based grouping below."

In [41]:
dns_columns = ['ts', 'uid', 'id.orig_h', 'id.orig_p', 'id.resp_h', 'id.resp_p',
               'proto', 'trans_id', 'rtt', 'query', 'qclass', 'qclass_name',
               'qtype', 'qtype_name', 'rcode', 'rcode_name', 'AA', 'TC', 'RD',
               'RA', 'Z', 'answers', 'TTLs', 'rejected', 'opcode', 'opcode_name']
df2 = pd.read_csv("../data/raw/zeek-captures/dns.log",sep='\t',comment='#', names = dns_columns)

df2[['query','answers']]

,query,answers
0,_ipps._tcp.local,-
1,connectivity-check.ubuntu.com,"2620:2d:4000:1::1099,2620:2d:4002:1::1062,2620..."
2,aliz_pc,-
3,aliz_pc,-
4,aliz_pc,-
5,aliz_pc,-
6,connectivity-check.ubuntu.com,"91.189.91.98,91.189.91.61,91.189.91.57,91.189...."
7,connectivity-check.ubuntu.com,"2620:2d:4002:1::1061,2620:2d:4002:1::1058,2620..."
8,example.com,"104.20.23.154,172.66.147.243"
9,example.com,"2606:4700:10::ac42:93f3,2606:4700:10::6814:179a"
